In [9]:
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression



from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# ⚠️ External libraries (install if needed)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
df = pd.read_csv('D:\RESEARCH\Youth Tobacco Survey\data preparation\susceptibility.csv')
df.columns

In [ ]:
df.drop(columns=['FinalWgt','PSU','Stratum','Country_Year'], inplace=True)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df = df[:100000]
df['Susceptibility'].value_counts()

In [ ]:
X = df.drop(columns=['Susceptibility'])
y = df['Susceptibility'].map({'non-susceptible': 0, 'susceptible': 1})

In [ ]:
X.columns

In [ ]:
categorical_cols = ['q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q18',
       'q19', 'q20', 'q21', 'q22', 'q23', 'Country', 'Region', 'Income Group', 'q27']
numeric_cols = ['q4', 'q5', 'q6', 'Year']

In [ ]:
num_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])


preprocessor = ColumnTransformer([
    ("num", num_pipeline, numeric_cols),
    ("cat", cat_pipeline, categorical_cols)
])

In [ ]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression()) # placeholder
])

param_grid = [

    # ---------------- Logistic Regression
    {
        "model": [LogisticRegression(max_iter=1000)],
        "model__C": [0.1, 1, 10]
    },

    # ---------------- KNN
    {
        "model": [KNeighborsClassifier()],
        "model__n_neighbors": [3, 5, 7, 9],
        "model__weights": ["uniform", "distance"]
    },

    # ---------------- Decision Tree
    {
        "model": [DecisionTreeClassifier()],
        "model__max_depth": [None, 5, 10, 20],
        "model__min_samples_split": [2, 5, 10]
    },

    # ---------------- Random Forest
    {
        "model": [RandomForestClassifier()],
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10, 20]
    },

    # ---------------- SVM
    {
        "model": [SVC()],
        "model__C": [0.1, 1, 10],
        "model__kernel": ["linear", "rbf"]
    },

    # ---------------- XGBoost
    {
        "model": [XGBClassifier(tree_method="gpu_hist",predictor="gpu_predictor",eval_metric="logloss")],
        "model__n_estimators": [100, 200],
        "model__max_depth": [3, 6],
        "model__learning_rate": [0.05, 0.1]
    },

    # ---------------- CatBoost
    {
        "model": [CatBoostClassifier(verbose=0)],
        "model__depth": [4, 6, 8],
        "model__learning_rate": [0.03, 0.1],
        "model__iterations": [100, 200]
    }
]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
print("Test accuracy:", pipeline.score(X_test, y_test))

In [ ]:
from sklearn.model_selection import cross_validate


scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}



grid = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_grid,
    cv=3,
    scoring=scoring,
    refit='roc_auc',
    n_jobs=2,
    n_iter=10,
    random_state=42
)

grid.fit(X, y)

print("Best Params:", grid.best_params_)
print("Best Accuracy:", grid.cv_results_["mean_test_accuracy"][grid.best_index_])
print("Best Precision:", grid.cv_results_["mean_test_precision"][grid.best_index_])
print("Best Recall:", grid.cv_results_["mean_test_recall"][grid.best_index_])
print("Best F1:", grid.cv_results_["mean_test_f1"][grid.best_index_])
print("Best AUC:", grid.cv_results_["mean_test_roc_auc"][grid.best_index_])